# Análisis de los eventos UNGRD 

**Autor:** Andres Felipe Sierra

**Objetivo:** Análisis de los datos de eventos UNGRD para la observación de las inundaciones y/o deslizamientos.

**Apreciaciones:** Este script tiene la implementación de CRISP-DM con los respectivos ciclos siendo: Bussiness understanding, Data understanding, Data preparation, Modeling, Evaluation y Deployment. 

## Bussiness understanding

UNGRD posee los datos de eventos catastroficos de Colombia, particularmente se necesita la información de recurrencia de los eventos por año en cuanto a las inundaciones y deslizamientos, por ende se realiza la preparación de los datos y su análisis para ver su viabilidad.

Esto nos ayuda a obtener un vector de conteo de los eventos por mes; Este vector posiblemente nos ayude a la referencia y tener en cuenta los parámetros para el modelo LSTM.

## Library

In [177]:
import pandas as pd
from IPython.display import display
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Funciones

In [154]:
def no_limits_display():
    pd.set_option("display.max_rows", None)        # todas las filas
    pd.set_option("display.max_columns", None)     # todas las columnas (si aplica)
    pd.set_option("display.width", 200)            # ancho
    pd.set_option("display.max_colwidth", None)    # texto completo

In [155]:
# Función para la obtención en un archivo .csv
def get_datos_csv(dat):
    df = pd.read_csv(dat)
    return df

In [156]:
# Función para la bsuqueda de duplicados y eliminación
def datos_duplicados(dat):
    print('Tamaño del dataset original: ', dat.shape)
    da = dat.drop_duplicates(keep='first', inplace=False)
    print('Tamaño del dataset sin duplicados: ',da.shape)
    return da

In [ ]:
# Función para la revisión de los valores nulos
def datos_nulos(dat, solo_nulos=False, ordenar=True):
    nulos = dat.isna().sum()
    pct = (nulos / len(dat) * 100).round(2)

    out = pd.DataFrame({
        "nulos": nulos,
        "% nulos": pct,
        "tipo": dat.dtypes.astype(str)
    })

    if solo_nulos:
        out = out[out["nulos"] > 0]

    if ordenar:
        out = out.sort_values("nulos", ascending=False)

    display(out)

In [158]:
# Función para la búsqueda de columnas conflictivas para análizar máximos
def columnas_conflictivas(dat):
    bad_cols = [c for c in dat.columns if dat[c].map(type).nunique() > 1]
    display(bad_cols)

In [159]:
# Función de conversión de columnas srt u object a numericas
def cambiar_numero(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip() # Pasa a todo string

    s = s.replace({"": None, "nan": None, "None": None, "NA": None, "N/A": None, "-": None}) # Normaliza "vacíos" comunes a NaN

    s = s.str.replace(r"[^\d\.,\-]", "", regex=True) # Elimina separadores de miles y caracteres no numéricos (deja dígitos, punto, coma y signo -)

    s = s.str.replace(",", ".", regex=False) # Si vienen con coma decimal estilo "12,5" -> "12.5"

    return pd.to_numeric(s, errors="coerce") # Convierte a número

## Data understanding

### Importación de datos

In [160]:
no_limits_display

<function __main__.no_limits_display()>

In [161]:
# Cargue de la bases de datos
df_nombre = '../Data/Emergencias_UNGRD._20260130.csv' # Se nombra la dirección de los datos

df = get_datos_csv(df_nombre)

C:\Users\sierr\AppData\Local\Temp\ipykernel_28080\1986087520.py:3: DtypeWarning: Columns (0: CANTIDAD RACIONES DE CAMPAÑA, 1: CANTIDAD PLASTICO NEGRO, 2: CANTIDAD BIG BAG, 3: CANTIDAD TEJAS DE FIBROCEMENTO) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dat)


### Exploración de datos

In [162]:
# Observamos la cabecera de los datos
df.head(3)

,FECHA,DEPARTAMENTO,MUNICIPIO,EVENTO,DIVIPOLA,FALLECIDOS,HERIDOS,DESAPARECIDOS,PERSONAS,FAMILIAS,VIVIENDAS DESTRUIDAS,VIVIENDAS AVERIADAS,VIAS AVERIADAS,PUENTES VEHICULARES,PUENTES PEATONALES,ACUEDUCTO,ALCANTARILLADO,CENTROS DE SALUD,CENTROS EDUCATIVOS,CENTROS COMUNITARIOS,HECTAREAS,OTROS-AFECTACION,SUBSIDIO DE ARRIENDO,ASISTENCIA NO ALIMENTARIA,APOYO ALIMENTARIO,MATERIALES CONSTRUCCION,SACOS - BIGBAG,OBRAS DE EMERGENCIA,CARROTANQUES - MOTOBOMBAS-PLANTA POTABILIZADORA,HORAS MAQUINA\nRETROEXCAVADORA\nBULLDOCER\nINTERVENTORIA,APOYO AEREO / TERRESTRE,FIC / TRANSFERENCIAS ECONOMICAS,INFRAESCTRUCTURA TECNOLOGICA,RECURSOS EJECUTADOS,OTROS,CANTIDAD KIT DE ALIMENTO,VALOR KIT DE ALIMENTO,CANTIDAD RACIONES DE CAMPAÑA,VALOR RACIONES DE CAMPAÑA,CANTIDAD KIT ASEO,VALOR KIT ASEO,CANTIDAD KIT COCINA,VALOR KIT COCINA,CANTIDAD COLCHONETA,VALOR COLCHONETA,CANTIDAD FRAZADAS/\nSOBRECAMAS,VALOR FRAZADAS/\nSOBRECAMAS,CANTIDAD SABANAS / COBIJA SENCILLA,VALOR SABANAS / COBIJA SENCILLA,CANTIDAD HAMACAS,VALOR HAMACAS,CANTIDAD TOLDILLOS,VALOR TOLDILLOS,CANTIDAD CARPAS,VALOR CARPAS,VALOR TOTAL ASISTENCIA NO ALIMENTARIA,CANTIDAD PLASTICO NEGRO,VALOR PLASTICO NEGRO,CANTIDAD SACOS,VALOR SACOS,CANTIDAD BIG BAG,VALOR BIG BAG,CANTIDAD CEMENTO,VALOR CEMENTO,CANTIDAD TEJAS DE ZINC,VALOR TEJAS DE ZINC,CANTIDAD TEJAS DE FIBROCEMENTO,VALOR TEJAS DE FIBROCEMENTO,DESCRIPCION MATERIALES DE CONSTRUCCION,VALOR MATERIALES DE CONSTRUCCION,VALOR TOTAL APOYO DEL FNGRD
0,2019 Jan 01 12:00:00 AM,ANTIOQUIA,SAN JERONIMO,INCENDIO ESTRUCTURAL,"5,656",0,0,0,5,1,1,0,0,0,0,0,0,0,0,0,0,No registra,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,No registra,$ - 0,$ - 0
1,2019 Jan 01 12:00:00 AM,RISARALDA,MARSELLA,INMERSION,"66,440",1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,No registra,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,No registra,$ - 0,$ - 0
2,2019 Jan 01 12:00:00 AM,ANTIOQUIA,ANZA,INCENDIO DE COBERTURA VEGETAL,"5,044",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,No registra,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,No registra,$ - 0,$ - 0


In [163]:
# Se muestra la cantidad de filas y la cantidad de columnas
print('El dataset consta de ', df.shape, ' (FILAS/COLUMNAS)')

El dataset consta de  (25857, 71)  (FILAS/COLUMNAS)


In [164]:
# Se busca cómo está ordenado el índice de los registros
print('El indie se compone por: ', df.index)

El indie se compone por:  RangeIndex(start=0, stop=25857, step=1)


In [165]:
# Se muestra el nombre de las columnas/variables
df.columns

Index(['FECHA', 'DEPARTAMENTO', 'MUNICIPIO', 'EVENTO', 'DIVIPOLA', 'FALLECIDOS', 'HERIDOS', 'DESAPARECIDOS', 'PERSONAS', 'FAMILIAS', 'VIVIENDAS DESTRUIDAS', 'VIVIENDAS AVERIADAS', 'VIAS AVERIADAS',
       'PUENTES VEHICULARES', 'PUENTES PEATONALES', 'ACUEDUCTO', 'ALCANTARILLADO', 'CENTROS DE SALUD', 'CENTROS EDUCATIVOS', 'CENTROS COMUNITARIOS', 'HECTAREAS', 'OTROS-AFECTACION',
       'SUBSIDIO DE ARRIENDO', 'ASISTENCIA NO ALIMENTARIA', 'APOYO ALIMENTARIO', 'MATERIALES CONSTRUCCION', 'SACOS - BIGBAG', 'OBRAS DE EMERGENCIA', 'CARROTANQUES - MOTOBOMBAS-PLANTA POTABILIZADORA',
       'HORAS MAQUINA\nRETROEXCAVADORA\nBULLDOCER\nINTERVENTORIA', 'APOYO AEREO / TERRESTRE', 'FIC / TRANSFERENCIAS ECONOMICAS', 'INFRAESCTRUCTURA TECNOLOGICA', 'RECURSOS EJECUTADOS', 'OTROS',
       'CANTIDAD KIT DE ALIMENTO', 'VALOR KIT DE ALIMENTO', 'CANTIDAD RACIONES DE CAMPAÑA', 'VALOR RACIONES DE CAMPAÑA', 'CANTIDAD KIT ASEO', 'VALOR KIT ASEO', 'CANTIDAD KIT COCINA',
       'VALOR KIT COCINA', 'CANTIDAD COLCHON

In [166]:
# Se muestra los arreglos
df.values

array([['2019 Jan 01 12:00:00 AM', 'ANTIOQUIA', 'SAN JERONIMO', ...,
        'No registra', '$    - 0', '$    - 0'],
       ['2019 Jan 01 12:00:00 AM', 'RISARALDA', 'MARSELLA', ...,
        'No registra', '$    - 0', '$    - 0'],
       ['2019 Jan 01 12:00:00 AM', 'ANTIOQUIA', 'ANZA', ...,
        'No registra', '$    - 0', '$    - 0'],
       ...,
       ['2022 Sep 10 12:00:00 AM', 'TOLIMA', 'VENADILLO', ...,
        'No registra', '$    - 0', '$    - 0'],
       ['2022 Sep 09 12:00:00 AM', 'HUILA', 'HOBO', ..., 'No registra',
        '$    - 0', '$    - 0'],
       ['2022 Sep 09 12:00:00 AM', 'CAUCA', 'EL TAMBO', ...,
        'No registra', '$    - 0', '$    - 0']],
      shape=(25857, 71), dtype=object)

In [167]:
# Tipos de datos con la respectiva cantidad de valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25857 entries, 0 to 25856
Data columns (total 71 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   FECHA                                                  25857 non-null  str   
 1   DEPARTAMENTO                                           25857 non-null  str   
 2   MUNICIPIO                                              25857 non-null  str   
 3   EVENTO                                                 25857 non-null  str   
 4   DIVIPOLA                                               25857 non-null  str   
 5   FALLECIDOS                                             25857 non-null  int64 
 6   HERIDOS                                                25857 non-null  int64 
 7   DESAPARECIDOS                                          25857 non-null  int64 
 8   PERSONAS                                               25857 non-nu

Hasta ahora, podemos observar que hay un total de 71 columnas/variables, con más de 25.000 registros, lo que es viable al ser una muestra grande para su análisis y obtener el conteo, ya veremos si extrayendo los demás datos sería suficiente.

Por otro lado, hay un total de 56 datos str(string - Cadena de caracteres), además datos int(entero - números mayores o iguales a cero) totales en 11, la última categoría de de datos es object (booleano) 4 totales.

### Búsqueda de registros duplicados

In [168]:
# Se busca los registros repetidos y cocatenarlos
df = datos_duplicados(df)

Tamaño del dataset original:  (25857, 71)
Tamaño del dataset sin duplicados:  (22031, 71)


Según lo observado, podemos ver que había datos duplicados con un total de 3826 registros duplicados.

### Busqueda de valores nulos

In [169]:
# Buscamos valores nulos/vacios por columna
datos_nulos(df,True)

,nulos,% nulos,tipo
CANTIDAD TEJAS DE FIBROCEMENTO,11,0.05,object


,nulos,% nulos,tipo
CANTIDAD TEJAS DE FIBROCEMENTO,11,0.05,object


Como se puede observar los valores nulos solo se ven representados en la columna "CANTIDAD TEJAS DE FIBROCEMENTO" y tampoco figura con un gran porcentaje de nulos, ergo, un cambio no denota repercusiones al ignorar, además que esta variable es de no importancia para nuestro objetivo.

Por otro lado, se valorará que variables necesitaremos en para el análisis.

### Busqueda y manipulación de datos extremos

Dato a problemas al intentar indagar algunas columnas, se decide buscar las variables problematicas y solucionarlo.

In [ ]:
# Se copia la base de datos original para evitar problemas
df_cp = df.copy()

In [171]:
# Buscamos las columnas conflictivas
columnas_conflictivas(df)

['CANTIDAD RACIONES DE CAMPAÑA',
 'CANTIDAD PLASTICO NEGRO',
 'CANTIDAD BIG BAG',
 'CANTIDAD TEJAS DE FIBROCEMENTO']

In [172]:
# Ya idenficado las variables conflitivas las convertimos en númericas
cols_mixtas = [
    "CANTIDAD RACIONES DE CAMPAÑA",
    "CANTIDAD PLASTICO NEGRO",
    "CANTIDAD BIG BAG",
    "CANTIDAD TEJAS DE FIBROCEMENTO",
]

df[cols_mixtas] = df[cols_mixtas].apply(cambiar_numero)

In [173]:
# Observamos los valores máximos
df.max()

FECHA                                                                                     2022 Sep 30 12:00:00 AM
DEPARTAMENTO                                                                                              VICHADA
MUNICIPIO                                                                                                 cachira
EVENTO                                                                                                   Vendaval
DIVIPOLA                                                                                                   99,773
FALLECIDOS                                                                                                     49
HERIDOS                                                                                                       313
DESAPARECIDOS                                                                                                   7
PERSONAS                                                                                

In [174]:
# Observamos los valores mínimos
df.min()

FECHA                                                       2019 Apr 01 12:00:00 AM
DEPARTAMENTO                                                               AMAZONAS
MUNICIPIO                                                                 ABEJORRAL
EVENTO                                                                    ACCIDENTE
DIVIPOLA                                                                          0
FALLECIDOS                                                                        0
HERIDOS                                                                           0
DESAPARECIDOS                                                                     0
PERSONAS                                                                          0
FAMILIAS                                                                          0
VIVIENDAS DESTRUIDAS                                                              0
VIVIENDAS AVERIADAS                                                         

Observando los datos tienen valores acordes a valores de pesos colombianos, sin embargo, al no poseer inteligencia del negocio se decide no alterar ninguna cambio teniendo en cuenta que necesitamos un conteo, pero por si necesitamos análizar otras cosas se decide mantener algunas columnas.

## Data preparation

### Filtración de los datos

In [ ]:
# Buscamos los valores unicos para asegurarnos los nombres de nuestro objetivo (Inundacion y deslizamiento)
df['EVENTO'].unique()

<StringArray>
[                       'INCENDIO ESTRUCTURAL',                                   'INMERSION',               'INCENDIO DE COBERTURA VEGETAL',                         'COLAPSO ESTRUCTURAL',
                          'MOVIMIENTO EN MASA',                            'ACCIDENTE MINERO',              'ACCIDENTE TRANSPORTE TERRESTRE',                                   'GRANIZADA',
     'ACCIDENTE TRANSPORTE MARITIMO O FLUVIAL',                                    'VENDAVAL',                                  'INUNDACION',                                       'OTROS',
                                     'EROSION',                                   'EXPLOSION',                     'AGLOMERACIÓN DE PÚBLICO',                        'ACTIVACIÓN VOLCANICA',
                                        'FUGA',                       'ACCIDENTE TECNOLOGICO',                          'AVENIDA TORRENCIAL',                                     'DERRAME',
                                    'INCE

In [179]:
df[(df['EVENTO'] == 'INUNDACIÓN')]

,FECHA,DEPARTAMENTO,MUNICIPIO,EVENTO,DIVIPOLA,FALLECIDOS,HERIDOS,DESAPARECIDOS,PERSONAS,FAMILIAS,VIVIENDAS DESTRUIDAS,VIVIENDAS AVERIADAS,VIAS AVERIADAS,PUENTES VEHICULARES,PUENTES PEATONALES,ACUEDUCTO,ALCANTARILLADO,CENTROS DE SALUD,CENTROS EDUCATIVOS,CENTROS COMUNITARIOS,HECTAREAS,OTROS-AFECTACION,SUBSIDIO DE ARRIENDO,ASISTENCIA NO ALIMENTARIA,APOYO ALIMENTARIO,MATERIALES CONSTRUCCION,SACOS - BIGBAG,OBRAS DE EMERGENCIA,CARROTANQUES - MOTOBOMBAS-PLANTA POTABILIZADORA,HORAS MAQUINA\nRETROEXCAVADORA\nBULLDOCER\nINTERVENTORIA,APOYO AEREO / TERRESTRE,FIC / TRANSFERENCIAS ECONOMICAS,INFRAESCTRUCTURA TECNOLOGICA,RECURSOS EJECUTADOS,OTROS,CANTIDAD KIT DE ALIMENTO,VALOR KIT DE ALIMENTO,CANTIDAD RACIONES DE CAMPAÑA,VALOR RACIONES DE CAMPAÑA,CANTIDAD KIT ASEO,VALOR KIT ASEO,CANTIDAD KIT COCINA,VALOR KIT COCINA,CANTIDAD COLCHONETA,VALOR COLCHONETA,CANTIDAD FRAZADAS/\nSOBRECAMAS,VALOR FRAZADAS/\nSOBRECAMAS,CANTIDAD SABANAS / COBIJA SENCILLA,VALOR SABANAS / COBIJA SENCILLA,CANTIDAD HAMACAS,VALOR HAMACAS,CANTIDAD TOLDILLOS,VALOR TOLDILLOS,CANTIDAD CARPAS,VALOR CARPAS,VALOR TOTAL ASISTENCIA NO ALIMENTARIA,CANTIDAD PLASTICO NEGRO,VALOR PLASTICO NEGRO,CANTIDAD SACOS,VALOR SACOS,CANTIDAD BIG BAG,VALOR BIG BAG,CANTIDAD CEMENTO,VALOR CEMENTO,CANTIDAD TEJAS DE ZINC,VALOR TEJAS DE ZINC,CANTIDAD TEJAS DE FIBROCEMENTO,VALOR TEJAS DE FIBROCEMENTO,DESCRIPCION MATERIALES DE CONSTRUCCION,VALOR MATERIALES DE CONSTRUCCION,VALOR TOTAL APOYO DEL FNGRD
4426,2019 Dec 29 12:00:00 AM,CHOCO,UNGUIA,INUNDACIÓN,"27,800",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,No registra,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,0,$ - 0,0.0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,$ - 0,0.0,$ - 0,0,$ - 0,0.0,$ - 0,0,$ - 0,0,$ - 0,0.0,$ - 0,No registra,$ - 0,$ - 0
4434,2019 Dec 31 12:00:00 AM,CAUCA,LOPEZ DE MICAY,INUNDACIÓN,"19,418",0,0,0,40,8,8,0,0,0,0,0,0,0,0,0,0,No registra,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,0,$ - 0,0.0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,$ - 0,0.0,$ - 0,0,$ - 0,0.0,$ - 0,0,$ - 0,0,$ - 0,0.0,$ - 0,No registra,$ - 0,$ - 0
4435,2019 Dec 31 12:00:00 AM,VALLE DEL CAUCA,BUENAVENTURA,INUNDACIÓN,"76,109",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,No registra,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,$ - 0,0,$ - 0,0.0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,0,$ - 0,$ - 0,0.0,$ - 0,0,$ - 0,0.0,$ - 0,0,$ - 0,0,$ - 0,0.0,$ - 0,No registra,$ - 0,$ - 0


Este script fue realizado para verificar una nueva información suminsitrada por UNGRD en Datos Abiertos, sin embargo, no es lo suficientemente solido y los datos anteriormnete ya preparados forman un dataset suficientemente robusto para su uso descartando este.